In [ ]:
# ============================================================
# DEEP_DIVE_YUKAWA_V2: ГЛУБОКИЙ РАСЧЁТ ЮКАВЫ (2.5 МЛН ШАГОВ)
# ============================================================
# REM: Этот ноутбук считает 4 "вкусные" точки для МЯГКОГО поля ЮКАВЫ.
# REM: Взаимодействие: J(r) = J0 · exp(-mu·r) / r  (экранированный потенциал).
# REM: Энергия: S = -ΣΣ J0·exp(-mu·r)/r · N_i N_j + σ · Perimeter.
# REM: Внутри цикла стоит for _ in range(5), поэтому 500 000 × 5 = 2 500 000 реальных шагов.

# REM: Импорт numpy для работы с массивами и случайными числами.
import numpy as np
# REM: Импорт label из scipy для поиска связных кластеров.
from scipy.ndimage import label
# REM: Импорт matplotlib для отрисовки картинок.
import matplotlib
# REM: Включаем фоновый режим 'Agg', чтобы графики сохранялись в файлы, а не открывались в окнах.
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# REM: Импорт csv для записи таблиц с результатами.
import csv
# REM: Импорт time для замера времени выполнения расчётов.
import time
# REM: Импорт os для работы с файловой системой.
import os
# REM: Импорт display и Image для вывода картинок прямо в Jupyter.
from IPython.display import display, Image as IPImage

# ------------------------------------------------------------
# REM: ПАПКИ ДЛЯ РЕЗУЛЬТАТОВ И ЧЕКПОИНТОВ
# ------------------------------------------------------------
# REM: output_dir — корневая папка, куда пишутся все результаты.
output_dir = "DEEP_DIVE_YUKAWA_V2"
# REM: csv_dir — подпапка для таблиц с числами (CSV).
csv_dir = os.path.join(output_dir, "csv")
# REM: png_dir — подпапка для картинок решётки (PNG).
png_dir = os.path.join(output_dir, "png")
# REM: ckpt_dir — подпапка для чекпоинтов (сохранений сетки для восстановления после сбоя).
ckpt_dir = os.path.join(output_dir, "checkpoints")
# REM: Создаём все папки. exist_ok=True предотвращает ошибку, если папка уже существует.
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# ------------------------------------------------------------
# REM: ПАРАМЕТРЫ МОДЕЛИ
# ------------------------------------------------------------
# REM: L — размер решётки (100×100 = 10 000 узлов/плексонов).
L = 100
# REM: STEPS — число итераций ВНЕШНЕГО цикла.
# REM: ВАЖНО: внутри цикла стоит for _ in range(5), поэтому реальное число шагов = 500 000 × 5 = 2 500 000.
STEPS = 500_000
# REM: beta — обратная температура (1/kT). Отвечает за тепловой шум.
beta = 1.0
# REM: J0 — СИЛА ПРИТЯЖЕНИЯ ЮКАВЫ (амплитуда потенциала).
# REM: J0 = 3.0 — усилено, "чтобы притяжение победило энтропию".
J0 = 3.0
# REM: mu (мю) — ПАРАМЕТР ЭКРАНИРОВАНИЯ ЮКАВЫ (обратная длина экранирования, "масса поля").
# REM: Чем больше mu, тем быстрее сила затухает с расстоянием. mu = 0.5.
mu = 0.5
# REM: r_cutoff — РАДИУС ОБРЕЗАНИЯ. Взаимодействие считается только на расстоянии до 4 клеток.
# REM: Это аналог "5-го атома" в полимерной цепи: на 4-й клетке сила ≈ e^-2 ≈ 0.13.
r_cutoff = 4

# REM: POINTS — список "вкусных" точек (плотность, натяжение σ), выбранных по скринингу.
# REM: (0.10, 2.0)  — 10% плотности, σ=2.0  — бинодальная зона, капли.
# REM: (0.10, 10.0) — 10% плотности, σ=10.0 — бинодаль при сильном натяжении.
# REM: (0.50, 1.5)  — 50% плотности, σ=1.5  — спинодальная зона, гель.
# REM: (0.50, 2.0)  — 50% плотности, σ=2.0  — стабильный гель.
POINTS = [
    (0.10, 2.0),
    (0.10, 10.0),
    (0.50, 1.5),
    (0.50, 2.0)
]

# REM: csv_file — путь к таблице результатов.
csv_file = os.path.join(csv_dir, "deep_dive_yukawa_v2.csv")

# REM: Если файла ещё нет — создаём его и записываем заголовки колонок.
if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['p_init', 'J0', 'sigma', 'step', 'density', 'P_inf', 'perimeter', 'max_size'])

# ------------------------------------------------------------
# REM: ФИЗИЧЕСКИЕ ФУНКЦИИ
# ------------------------------------------------------------

# REM: init_lattice — создаёт случайную решётку: с вероятностью p узел занят (1 = плексон), иначе пуст (0).
def init_lattice(L, p):
    return (np.random.random((L, L)) < p).astype(int)

# REM: make_yukawa_kernel — строит МАТРИЦУ (ядро) сил Юкавы размером (2*r_cut+1) × (2*r_cut+1).
# REM: В каждой клетке ядра хранится сила J(r) = J0 · exp(-mu·r) / r.
# REM: Ядро считается ОДИН раз (это как таблица синусов), чтобы не считать экспоненту на каждом шаге.
def make_yukawa_kernel(r_cut, mu, J0_val):
    size = 2 * r_cut + 1
    kernel = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            # REM: dx, dy — смещение от центра ядра.
            dx, dy = i - r_cut, j - r_cut
            r = np.sqrt(dx**2 + dy**2)
            # REM: в центре (r=0) силы нет (иначе деление на ноль); считаем только до r_cut.
            if r > 0 and r <= r_cut:
                kernel[i, j] = J0_val * np.exp(-mu * r) / r
    return kernel

# REM: get_local_window_torus — вырезает ОКНО (2*r_cut+1) вокруг узла (i,j) с учётом тора.
# REM: Благодаря операции % L (остаток от деления) границы зациклены: узел "видит" себя через край.
def get_local_window_torus(grid, i, j, r_cut):
    L = grid.shape[0]
    window = np.zeros((2*r_cut+1, 2*r_cut+1))
    for di in range(-r_cut, r_cut+1):
        for dj in range(-r_cut, r_cut+1):
            window[di + r_cut, dj + r_cut] = grid[(i + di) % L, (j + dj) % L]
    return window

# REM: local_energy — энергия взаимодействия ОДНОГО занятого узла с его окружением.
# REM: E = -Σ(окно · ядро). Знак минус означает, что взаимодействие ПРИТЯГИВАЕТ (снижает энергию).
def local_energy(grid, i, j, kernel, r_cut):
    win = get_local_window_torus(grid, i, j, r_cut)
    return -np.sum(win * kernel)

# REM: get_local_perimeter_both — ЛОКАЛЬНЫЙ периметр вокруг двух обмениваемых узлов.
# REM: Считается только в маленьком окне вокруг (i1,j1) и (i2,j2) — это в разы быстрее полного периметра.
# REM: Периметр = число рёбер, где занятая клетка (1) касается пустой (0).
def get_local_perimeter_both(grid, i1, j1, i2, j2, radius=2):
    L = grid.shape[0]
    p = 0
    min_i, max_i = min(i1, i2) - radius, max(i1, i2) + radius
    min_j, max_j = min(j1, j2) - radius, max(j1, j2) + radius
    for i in range(min_i, max_i + 1):
        for j in range(min_j, max_j + 1):
            ni, nj = i % L, j % L
            if grid[ni, nj] == 1:
                if grid[(ni-1)%L, nj] == 0: p += 1
                if grid[(ni+1)%L, nj] == 0: p += 1
                if grid[ni, (nj-1)%L] == 0: p += 1
                if grid[ni, (nj+1)%L] == 0: p += 1
    return p

# REM: metropolis_step_yukawa — ОДИН шаг Метрополиса для поля Юкавы.
# REM: Обмен занятого и пустого узла сохраняет плотность.
# REM: ВАЖНО: энергия считается ТОЛЬКО вокруг занятого узла (пустой узел энергии не даёт),
# REM: поэтому двойного счёта энергии нет (исправлено в ранних версиях кода).
def metropolis_step_yukawa(grid, sigma, yukawa_kernel, beta=1.0):
    L = grid.shape[0]
    # REM: случайно ищем ЗАНЯТЫЙ узел (1).
    while True:
        i1, j1 = np.random.randint(0, L, 2)
        if grid[i1, j1] == 1: break
    # REM: случайно ищем ПУСТОЙ узел (0).
    while True:
        i2, j2 = np.random.randint(0, L, 2)
        if grid[i2, j2] == 0: break

    # REM: энергия ДО (считаем только занятый i1, т.к. i2 пуст).
    E_old = local_energy(grid, i1, j1, yukawa_kernel, r_cutoff)
    P_old = get_local_perimeter_both(grid, i1, j1, i2, j2)

    # REM: обмен узлов.
    grid[i1, j1] = 0
    grid[i2, j2] = 1

    # REM: энергия ПОСЛЕ (считаем только занятый i2, т.к. i1 теперь пуст).
    E_new = local_energy(grid, i2, j2, yukawa_kernel, r_cutoff)
    P_new = get_local_perimeter_both(grid, i1, j1, i2, j2)

    # REM: полное изменение энергии = (разница энергий Юкавы) + sigma · (разница периметров).
    delta_S = (E_new - E_old) + sigma * (P_new - P_old)

    # REM: критерий Метрополиса: принимаем, если энергия упала, или с вероятностью exp(-βΔS).
    if delta_S <= 0 or np.random.random() < np.exp(-beta * delta_S):
        return True
    else:
        # REM: если не приняли — возвращаем узлы на место.
        grid[i1, j1] = 1
        grid[i2, j2] = 0
        return False

# REM: measure — измеряет макроскопические характеристики системы.
# REM: P_inf — доля узлов в САМОМ БОЛЬШОМ кластере (0 = рассеяно, 1 = монолит).
# REM: density — реальная измеренная плотность.
# REM: perimeter — периметр (граница материя/пустота).
# REM: max_size — абсолютный размер наибольшего кластера.
def measure(grid):
    padded = np.pad(grid, pad_width=1, mode='wrap')
    labeled, n = label(padded, structure=[[0,1,0],[1,1,1],[0,1,0]])
    labeled = labeled[1:-1, 1:-1]
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    sizes = sizes[sizes > 0]
    if len(sizes) == 0: return None

    P_inf = float(sizes.max() / np.sum(sizes))
    density = float(np.mean(grid))
    right = np.roll(grid, 1, axis=1); left = np.roll(grid, -1, axis=1)
    up = np.roll(grid, 1, axis=0); down = np.roll(grid, -1, axis=0)
    perimeter = int(np.sum((grid == 1) & (right == 0)) + np.sum((grid == 1) & (left == 0)) +
                    np.sum((grid == 1) & (up == 0)) + np.sum((grid == 1) & (down == 0)))

    return {'density': density, 'P_inf': P_inf, 'perimeter': perimeter, 'max_size': int(sizes.max())}

# ------------------------------------------------------------
# REM: ЗАПУСК С ПОДДЕРЖКОЙ ВОССТАНОВЛЕНИЯ (ЧЕКПОИНТЫ)
# ------------------------------------------------------------
# REM: Создаём ядро Юкавы ОДИН раз перед запуском (для всех точек).
yukawa_kernel = make_yukawa_kernel(r_cutoff, mu, J0)

# REM: Перебираем все 4 точки.
for p, sigma in POINTS:
    # REM: пути к файлам чекпоинтов (сохранённая сетка и номер шага).
    ckpt_grid_file = os.path.join(ckpt_dir, f"grid_p{p}_s{sigma}.npy")
    ckpt_step_file = os.path.join(ckpt_dir, f"step_p{p}_s{sigma}.npy")

    start_step = 0
    grid = None

    # REM: если есть сохранение — восстанавливаемся с него.
    if os.path.exists(ckpt_grid_file) and os.path.exists(ckpt_step_file):
        grid = np.load(ckpt_grid_file)
        start_step = int(np.load(ckpt_step_file)[0])
        print(f"\n[RESTART] Плотность {p*100}%, Сигма {sigma}. Восстанавливаем с шага {start_step*5}...", flush=True)
    elif os.path.exists(csv_file):
        # REM: если сохранения нет — проверяем, не досчитана ли точка (по финальной строке в CSV).
        is_done = False
        with open(csv_file, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row) >= 4 and abs(float(row[0]) - p) < 1e-5 and abs(float(row[2]) - sigma) < 1e-5:
                    if int(row[3]) == STEPS * 5: is_done = True
        if is_done:
            print(f"[SKIP] Плотность {p*100}%, Сигма {sigma} уже полностью посчитана.", flush=True)
            continue
        else:
            grid = init_lattice(L, p)
    else:
        grid = init_lattice(L, p)

    # REM: печатаем заголовок таблицы для текущей точки.
    print("\n" + "="*65, flush=True)
    print(f"СЧИТАЕМ (Юкава): Плотность = {p*100}%, Сигма = {sigma}, J0 = {J0}", flush=True)
    print("="*65, flush=True)
    print(f"{'Шаг':>8} | {'density':>8} | {'P_inf':>8} | {'Perimeter':>10} | {'Max_Size':>8} | {'Time(s)':>7}", flush=True)
    print("-" * 65, flush=True)

    # REM: засекаем время начала расчёта точки.
    t0 = time.time()

    # REM: основной цикл (начинается с start_step при восстановлении).
    for step in range(start_step, STEPS):
        # REM: внутренний цикл делает 5 шагов Метрополиса за 1 итерацию.
        for _ in range(5):
            metropolis_step_yukawa(grid, sigma, yukawa_kernel, beta)

        # REM: реальное число сделанных шагов = step * 5.
        real_step = step * 5

        # REM: каждые 250 000 реальных шагов — замер, запись в CSV, чекпоинт и картинка.
        if real_step % 250_000 == 0 and real_step > 0:
            m = measure(grid)
            elapsed = time.time() - t0

            # REM: записываем строку в CSV.
            with open(csv_file, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([p, J0, sigma, real_step, m['density'], m['P_inf'], m['perimeter'], m['max_size']])

            print(f"{real_step:>8} | {m['density']:>8.3f} | {m['P_inf']:>8.3f} | {m['perimeter']:>10.1f} | {m['max_size']:>8d} | {elapsed:>7.1f}", flush=True)

            # REM: сохраняем сетку и номер шага (чекпоинт).
            np.save(ckpt_grid_file, grid)
            np.save(ckpt_step_file, np.array([step]))

            # REM: рисуем и сохраняем картинку текущего состояния (чёрный = плексон, белый = пустота).
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(grid, cmap='binary', interpolation='nearest')
            ax.set_title(f'Yukawa D={p*100}%, σ={sigma}, Step={real_step}')
            img_path = os.path.join(png_dir, f"Y_D{p}_S{sigma}_step{real_step}.png")
            plt.savefig(img_path, dpi=100, bbox_inches='tight')
            plt.close(fig)
            display(IPImage(filename=img_path))

    # REM: после цикла — финальный замер и запись финальной строки.
    m = measure(grid)
    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([p, J0, sigma, STEPS*5, m['density'], m['P_inf'], m['perimeter'], m['max_size']])

    # REM: удаляем чекпоинты (точка полностью досчитана).
    if os.path.exists(ckpt_grid_file): os.remove(ckpt_grid_file)
    if os.path.exists(ckpt_step_file): os.remove(ckpt_step_file)

    print(f"ТОЧКА ЗАВЕРШЕНА.", flush=True)

print("\nВСЕ 4 ТОЧКИ ЮКАВЫ ГОТОВЫ! Данные в DEEP_DIVE_YUKAWA_V2.", flush=True)
